<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex11.2-energy-optimization/Ex11.2_20_energy_optimization_pinn.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->


# Ex_11.2b — The Same Optimisation, as a Neural Network

**Deep Learning for Engineering · Aalborg University**

Notebook 10 solved the energy-optimal speed profile with `scipy.optimize`. It
treated the profile as a vector of numbers and handed the whole thing to
L-BFGS-B.

This notebook solves the identical problem a different way: the profile becomes a
small neural network, the physics goes into a loss, and autograd supplies the
gradient. It is L7.1's machinery on a robot car.

**Read the comparison honestly.** For a problem this size scipy is the sensible
engineering choice, and notebook 10 remains the one you run on the car. This
notebook exists because the formulation generalises where scipy does not — to
many design variables, to constraints that are themselves learned, and to a
profile that must be a smooth function rather than a list of samples.

Run this off the car. It needs nothing but the parameters you identified in
notebook 10.

---

### What you will do

1. represent the speed profile as a network of arc length, not as a vector;
2. enforce the grip limit as a **layer**, so it cannot be violated during training;
3. write the lap energy as a loss and let autograd differentiate through it;
4. train with Adam then L-BFGS, and compare against notebook 10's answer.


## 0 · The parameters you identified

Copy these from notebook 10. Do not re-fit them here — the point of this notebook
is the optimisation, not the identification.


In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex11.2-energy-optimization/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)

# ── from notebook 10, section 4 ─────────────────────────────────────────────
MASS_KG      = 1.6      # <- your measured mass
C_RR         = 0.02     # <- your identified rolling coefficient
K_TORQUE     = 1.0      # <- your identified torque constant
R_ARM        = 1.5      # <- your identified armature resistance
P_HOTEL_W    = 4.0      # <- your measured hotel load
MU_MEASURED  = 0.55     # <- your measured friction coefficient
V_MAX_CAR    = 1.8      # <- your measured top speed at capped throttle
W_WEIGHT     = 0.0      # J per second: 0 = pure energy, >0 = trade time for energy

G = 9.81
print(f"mass {MASS_KG} kg | hotel {P_HOTEL_W} W | mu {MU_MEASURED}")

## 1 · The course, as arc length and curvature

Same survey you used in notebook 10. Replace the placeholder with your own.


In [ ]:
# PLACEHOLDER — replace with your course survey, 1/R per segment
DS = 0.5                                    # segment length, m
curvature = np.array([0.0]*6 + [0.9]*4 + [0.0]*8 + [1.4]*3 +
                     [0.0]*5 + [0.6]*4 + [0.0]*6, dtype=float)
N = len(curvature)
s = np.arange(N) * DS                       # arc length at each segment
LAP_LEN = N * DS

# the per-segment ceiling: grip limit, or the car's top speed, whichever is lower
v_lim = np.array([min(math.sqrt(MU_MEASURED*G/k), V_MAX_CAR) if k > 1e-6
                  else V_MAX_CAR for k in curvature])

print(f"{N} segments, {LAP_LEN:.1f} m lap")
print(f"ceiling: {v_lim.min():.2f} to {v_lim.max():.2f} m/s")

## 2 · The profile as a network, with the grip limit built in

Two decisions here, and both come from L7.1.

**The profile is a function, not a vector.** The network maps arc length to speed,
so the answer is defined everywhere on the lap rather than at the segments you
happened to choose. Refining the segments does not change the number of unknowns.

**The grip limit is a layer, not a penalty.** The raw output is squashed through a
sigmoid scaled to the local ceiling, so the constraint holds by construction — at
every training step and at every point, including points the optimiser never
sampled. That is L7.1's hard enforcement, applied to a speed instead of a boundary
condition.


In [ ]:
V_FLOOR = 0.2                               # m/s, below this the car stalls

class SpeedProfile(nn.Module):
    """v(s), bounded below by V_FLOOR and above by the local ceiling."""

    def __init__(self, width=48, depth=3):
        super().__init__()
        layers, d_in = [], 1
        for _ in range(depth):
            layers += [nn.Linear(d_in, width), nn.Tanh()]
            d_in = width
        layers += [nn.Linear(d_in, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, s_norm, v_ceiling):
        raw = self.net(s_norm)                       # unconstrained
        frac = torch.sigmoid(raw).squeeze(-1)        # in (0, 1)
        return V_FLOOR + frac * (v_ceiling - V_FLOOR)

model = SpeedProfile()
print(f"{sum(p.numel() for p in model.parameters())} parameters "
      f"for a {N}-segment profile")

## 3 · The lap energy, as a loss

This is the same power model as notebook 10, written so autograd can
differentiate through it.

Three terms in the power: copper loss quadratic in current, mechanical power at
the wheels, and the constant hotel load. The current follows from the tractive
force, which follows from rolling resistance plus the inertial term.

The one place to be careful is the acceleration. Notebook 10 used
`np.gradient`; here it must be a differentiable operation, so it is a finite
difference on the tensor.


In [ ]:
s_t     = torch.tensor(s / LAP_LEN, dtype=torch.float32).unsqueeze(-1)  # scaled input
v_cap_t = torch.tensor(v_lim, dtype=torch.float32)
ds_t    = torch.tensor(DS, dtype=torch.float32)

def lap_cost(v):
    """Energy over one lap, plus w * lap time. v: (N,) tensor in m/s."""
    dt = ds_t / v
    # dv/ds by central difference, then a = v dv/ds
    dv_ds = torch.gradient(v, spacing=(float(DS),))[0]
    a     = v * dv_ds
    F     = C_RR * MASS_KG * G + MASS_KG * a
    i     = torch.clamp(F / K_TORQUE, min=0.0)       # no regeneration
    P     = i**2 * R_ARM + F * v + P_HOTEL_W
    E     = torch.sum(P * dt)
    T     = torch.sum(dt)
    return E + W_WEIGHT * T, E, T

with torch.no_grad():
    v0 = model(s_t, v_cap_t)
    c0, e0, t0 = lap_cost(v0)
print(f"before training: {float(e0):.1f} J over {float(t0):.2f} s")

## 4 · Train — Adam, then L-BFGS

The same two-stage pattern as every PINN in this course. Adam is robust from a
poor start; L-BFGS is precise near the minimum and is what actually converges
this problem.

Watch the loss. It should fall quickly under Adam and then drop again when
L-BFGS takes over.


In [ ]:
history = []

opt = torch.optim.Adam(model.parameters(), lr=3e-3)
for step in range(1500):
    opt.zero_grad()
    v = model(s_t, v_cap_t)
    cost, E, T = lap_cost(v)
    cost.backward()
    opt.step()
    if step % 250 == 0:
        history.append(float(cost.detach()))
        print(f"  adam {step:5d}   cost {float(cost):8.2f}   E {float(E):7.1f} J")

lbfgs = torch.optim.LBFGS(model.parameters(), max_iter=300,
                          tolerance_grad=1e-9, line_search_fn="strong_wolfe")

def closure():
    lbfgs.zero_grad()
    cost, _, _ = lap_cost(model(s_t, v_cap_t))
    cost.backward()
    return cost

lbfgs.step(closure)

with torch.no_grad():
    v_pinn = model(s_t, v_cap_t)
    c_pinn, E_pinn, T_pinn = lap_cost(v_pinn)
print(f"\nafter L-BFGS: {float(E_pinn):.1f} J over {float(T_pinn):.2f} s")

## 5 · Against notebook 10

Run the scipy optimiser on the identical problem and compare. Three things to
look at, in order.

**The energy.** Expect the network to be a few percent worse, and expect that to
be reproducible rather than a convergence failure. The reason is the squashing
layer: a sigmoid approaches the ceiling asymptotically, so on a straight where
the optimum is to sit exactly at the limit, the network gets close and never
arrives. A bounded solver can sit on the bound exactly.

That is the price of hard enforcement by construction, and it is worth
recognising — L7.1 said hard enforcement was strictly better, and here is a case
where it costs something. If your gap is much larger than a few percent, then it
is a convergence problem.

**The profile shape.** Both should be fast on straights and slow in corners. The
network's will be smoother, because a small network cannot represent a jagged
function — which is a bias, and on this problem a helpful one.

**What each cost you.** Note the run times honestly.


In [ ]:
from scipy.optimize import minimize

def scipy_cost(v):
    v = np.clip(v, V_FLOOR, v_lim)
    dt = DS / v
    F = C_RR * MASS_KG * G + MASS_KG * np.gradient(v) * v / DS
    i = np.maximum(F / K_TORQUE, 0.0)
    P = i**2 * R_ARM + F * v + P_HOTEL_W
    return float(np.sum(P * dt) + W_WEIGHT * np.sum(dt))

res = minimize(scipy_cost, np.minimum(v_lim, 1.5), method="L-BFGS-B",
               bounds=[(V_FLOOR, float(u)) for u in v_lim])
v_scipy = np.clip(res.x, V_FLOOR, v_lim)
T_scipy = float(np.sum(DS / v_scipy))
E_scipy = scipy_cost(v_scipy) - W_WEIGHT * T_scipy

vp = v_pinn.detach().numpy()
print(f"  scipy : {E_scipy:7.1f} J   {T_scipy:5.2f} s")
print(f"  PINN  : {float(E_pinn):7.1f} J   {float(T_pinn):5.2f} s")
print(f"  gap   : {100*(float(E_pinn)-E_scipy)/E_scipy:+.2f} %")
print(f"  max ceiling violation, PINN: {float(np.max(vp - v_lim)):.2e} m/s")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(s, v_lim,    lw=1.2, ls="--", label="grip ceiling")
ax.plot(s, v_scipy,  lw=1.6, label="scipy L-BFGS-B")
ax.plot(s, vp,       lw=2.0, label="network + autograd")
ax.set_xlabel("arc length, m"); ax.set_ylabel("speed, m/s")
ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 6 · What to write up

Answer these in your report. They are the point of the notebook.

1. **How close were the two answers**, in joules and in percent?

2. **Did the network ever violate the grip ceiling?** It should not have, by
   construction. Say why that is a property of the architecture rather than of
   the training.

3. **Which took longer**, and by how much? Be honest — on this problem the
   classical optimiser is expected to win.

4. **Name one change to the problem that would reverse that.** Some candidates:
   many more design variables; a constraint that is itself learned from data; a
   profile that must be evaluated at arbitrary points rather than at segments; or
   optimising the vehicle parameters and the profile jointly.

5. The network has far more parameters than the profile has segments.
   **Why is that not overfitting?** The answer is the same one as L7.1 slide 9,
   and it is worth stating in your own words.

---

### Hand in

`Ex11.2b_<team>.json` with both energies, both run times, the maximum ceiling
violation, and your answers to 4 and 5.


In [ ]:
import json, time, platform

submission = {
    "team": "team-01",                       # <- your team
    "E_scipy_J":  round(float(E_scipy), 2),
    "T_scipy_s":  round(float(T_scipy), 3),
    "E_pinn_J":   round(float(E_pinn), 2),
    "T_pinn_s":   round(float(T_pinn), 3),
    "gap_percent": round(100*(float(E_pinn)-E_scipy)/E_scipy, 3),
    "max_ceiling_violation_mps": float(np.max(vp - v_lim)),
    "network": {"width": 48, "depth": 3,
                "params": sum(p.numel() for p in model.parameters())},
    "when": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "python": platform.python_version(),
}
with open(f"Ex11.2b_{submission['team']}.json", "w") as f:
    json.dump(submission, f, indent=2)
print(json.dumps(submission, indent=2))